In [1]:
import torch
from frameworks.LightenDiffusion.models.decom import ImageEncoder, ImageDecoder
%load_ext autoreload
%autoreload 2

In [7]:
# ---- Configuration ----
B = 1                   # batch size
H = W = 256             # input spatial size (must be divisible by 2**len(channel_factors))
base_channels = 64      # same as in your model defaults
channel_factors = [1, 2, 4]
in_channels = 3
encoded_channels = 64
out_channels = 3

# ---- Instantiate ----
encoder = ImageEncoder(
    base_channels=base_channels,
    channel_factors=channel_factors,
    in_channels=in_channels,
    encoded_channels=encoded_channels
)
decoder = ImageDecoder(
    base_channels=base_channels,
    channel_factors=channel_factors,
    out_channels=out_channels,
    encoded_channels=encoded_channels
)

# Move to CPU (or .cuda() if you prefer)
device = torch.device("cpu")
encoder.to(device)
decoder.to(device)

# ---- Forward pass through encoder ----
x = torch.randn(B, in_channels, H, W, device=device)
# Should return a flat tuple: (feat0, feat1, ..., encoded)
out = encoder(x)
assert isinstance(out, tuple), "Encoder must return a tuple"
*features, encoded = out

# Check number of pyramid levels
L = len(channel_factors)
assert len(features) == L, f"Expected {L} pyramid levels, got {len(features)}"

# Check each feature map’s shape
for i, feat in enumerate(features):
    expected_c = base_channels * channel_factors[i]
    expected_h = H // (2 ** (i+1))
    expected_w = W // (2 ** (i+1))
    actual = feat.shape
    assert actual == (B, expected_c, expected_h, expected_w), (
        f"features[{i}] shape {actual} != expected {(B, expected_c, expected_h, expected_w)}"
    )

# Check encoded shape
down_factor = 2 ** L
expected_encoded_shape = (B, encoded_channels, H // down_factor, W // down_factor)
assert encoded.shape == expected_encoded_shape, (
    f"Encoded shape {encoded.shape} != expected {expected_encoded_shape}"
)

print("✅ Encoder shapes OK")

# ---- Forward pass through decoder ----
# Pass latent first, then each skip feature in the same order
recon = decoder(encoded, *features)

# Check reconstruction shape
expected_recon_shape = (B, out_channels, H, W)
assert recon.shape == expected_recon_shape, (
    f"Reconstruction shape {recon.shape} != expected {expected_recon_shape}"
)

print("✅ Decoder shapes OK")
print("All shape checks passed.")


✅ Encoder shapes OK
✅ Decoder shapes OK
All shape checks passed.
